In [ ]:
import pandas as pd

client_train_data = pd.read_csv("data/train/client_train.csv", low_memory=False)
invoice_train_data = pd.read_csv("data/train/invoice_train.csv",  low_memory=False)

In [ ]:
# One row per client is what we predict on, so invoices get summarised per client further down.
# First, a few per-invoice columns that those summaries are built from.
invoices = invoice_train_data.copy()
invoices["invoice_date"] = pd.to_datetime(invoices["invoice_date"])
invoices["counter_type"] = invoices["counter_type"].map({"ELEC": 0, "GAZ": 1})
# counter_statue is a numeric code, except for 13 invoices holding the letter "A": those become missing
invoices["counter_statue"] = pd.to_numeric(invoices["counter_statue"], errors="coerce")
invoices["diff_index"] = invoices["new_index"] - invoices["old_index"]
# All four consumption levels added up per invoice
invoices["total_cons"] = invoices[["consommation_level_1", "consommation_level_2",
                                   "consommation_level_3", "consommation_level_4"]].sum(axis=1)
by_client = invoices.groupby("client_id")

## 1. The data

In [ ]:
# Two tables: one row per client (with the fraud label), one row per invoice
print("Clients:", client_train_data.shape, "| Invoices:", invoice_train_data.shape)
print("Clients that have invoices:", invoice_train_data["client_id"].nunique(), "of", len(client_train_data))
print("Invoices per client (min / median / max):",
      invoice_train_data.groupby("client_id").size().agg(["min", "median", "max"]).tolist())
display(client_train_data.head(3))
display(invoice_train_data.head(3))

## 2. Fraud is rare, so accuracy is the wrong yardstick

In [ ]:
import seaborn as sns
sns.countplot(data=client_train_data, x="target")
# We see a very imbalanced result

In [ ]:
import matplotlib.pyplot as plt

target_counts = client_train_data["target"].value_counts().sort_index()
plt.pie(
    target_counts,
    labels=target_counts.index,
    autopct="%1.1f%%"
)
plt.title("Client target distribution")
plt.show()

In [ ]:
# A model that never flags anyone already looks very accurate - that is why we do not judge by accuracy
no_fraud_accuracy = 1 - client_train_data["target"].mean()
print(f"Accuracy of always predicting 'no fraud': {no_fraud_accuracy:.1%}")

**AUC (area under the ROC curve), in plain words:** pick one fraud client and one honest client at random.
AUC is the chance that the model gives the fraud client the higher risk score.
0.5 = coin flip, 1.0 = perfect ranking. It rewards ranking risky clients first, which is what an inspection list needs.

In [ ]:
# One row per client: start from the client table, add invoice summaries below
client_data = client_train_data.set_index("client_id")
y = client_data["target"]
X = client_data[["client_catg", "disrict", "region", "creation_date"]].copy()

In [ ]:
from sklearn.model_selection import train_test_split

# One row per client, so a plain split keeps each client on one side only.
# 25% test set; stratify keeps the fraud ratio the same in both parts.
# Done once, up front, so every model below is scored on the same clients.
train_ids, test_ids = train_test_split(y.index, test_size=0.25, stratify=y, random_state=42)
y_train, y_test = y.loc[train_ids], y.loc[test_ids]

In [ ]:
X['client_catg'] = X['client_catg'].astype(int)
X['disrict'] = X['disrict'].astype(int)

X["creation_date"] = pd.to_datetime(X["creation_date"]).astype("int64")

## 3. Baseline: LightGBM on the raw data, no feature engineering

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score

# Raw data only: the client columns plus the average of every raw invoice column per client.
# (The model needs one row per client, so averaging is the one unavoidable roll-up - no new ideas added.)
raw_invoice_cols = ["tarif_type", "counter_statue", "counter_code", "reading_remarque", "counter_coefficient",
                    "consommation_level_1", "consommation_level_2", "consommation_level_3", "consommation_level_4",
                    "old_index", "new_index", "months_number", "counter_type"]
X_base = X.join(invoices.groupby("client_id")[raw_invoice_cols].mean())

results = []  # one row per model: train and test AUC, for the results table at the end
def record(name, model, X_all):
    results.append({
        "model": name,
        "train_auc": roc_auc_score(y_train, model.predict_proba(X_all.loc[train_ids])[:, 1]),
        "test_auc": roc_auc_score(y_test, model.predict_proba(X_all.loc[test_ids])[:, 1]),
    })

baseline = LGBMClassifier(random_state=42, verbose=-1).fit(X_base.loc[train_ids], y_train)
record("1. Baseline (raw data, default settings)", baseline, X_base)
print(results[-1])

## 4. Feature engineering

In [ ]:
# reading_remarque is a code the meter reader assigns at each reading. We use it as it comes (not engineered),
# but fraud rates differ a lot by value. Share of clients with at least one invoice carrying that code who are fraud:
pairs = invoices[["client_id", "reading_remarque"]].drop_duplicates().join(y, on="client_id")
remarque_table = pairs.groupby("reading_remarque")["target"].agg(n_clients="size", fraud_rate_pct=lambda s: round(s.mean() * 100, 2))
print(f"Overall fraud rate: {y.mean() * 100:.2f}%")
display(remarque_table.sort_values("fraud_rate_pct", ascending=False))

# Small helper used below: fraud rate per group of clients
def fraud_rate_by(groups, name):
    table = y.groupby(groups, observed=True).agg(n_clients="size", fraud_rate_pct=lambda s: round(s.mean() * 100, 2))
    table.index.name = name
    return table

In [ ]:
X = X_base.copy()  # engineered features are added on top of the raw baseline columns

In [ ]:
# If the result is negative, this might indicate a fraud. This is *drumroll* FEATURE ENGINEERING!
# Per client we keep the average difference between new and old meter index
X["mean_diff_index"] = by_client["diff_index"].mean()

In [ ]:
# How many different tariff types and meters does each client have across their invoices?
# (a client switching between tariffs could be a fraud signal), plus how many invoices they have.
# (Average months and share of gas are already in the raw baseline columns.)
X["n_tarif_types"] = by_client["tarif_type"].nunique()
X["n_counters"] = by_client["counter_number"].nunique()
X["n_invoices"] = by_client.size()

In [ ]:
display(fraud_rate_by(X["n_tarif_types"], "number of tariff types"))

In [ ]:
# Feature 1: consumption pattern per client (all four consumption levels added up per invoice).
# Fraudsters often under-report, so low, flat or zero consumption can stand out.
# (The average per consumption level is already in the raw baseline columns.)
X["cons_mean"] = by_client["total_cons"].mean()
X["cons_std"] = by_client["total_cons"].std().fillna(0)  # clients with one invoice have no spread
X["cons_min"] = by_client["total_cons"].min()
X["cons_zero_share"] = (invoices["total_cons"] == 0).groupby(invoices["client_id"]).mean()

In [ ]:
display(fraud_rate_by(X["cons_zero_share"].gt(0).map({False: "never zero", True: "has zero-consumption invoices"}), "zero consumption"))

In [ ]:
# Feature 2: negative index differences per client (the meter reading went backwards).
# Count and share, because 1 negative out of 1 invoice (100%) is not the same as 1 out of 50 (2%).
negative = invoices["diff_index"] < 0
X["n_negative_diff"] = negative.groupby(invoices["client_id"]).sum()
X["share_negative_diff"] = negative.groupby(invoices["client_id"]).mean()

In [ ]:
display(fraud_rate_by(pd.cut(X["n_negative_diff"], [-1, 0, 1, 5, float("inf")], labels=["0", "1", "2-5", "6+"]),
                      "invoices with index going backwards"))

In [ ]:
# Feature 3: consumption drop per client = (average of the later half of invoices - average of the earlier half)
# relative to the earlier half. A sudden drop is the classic "tampered meter" shape.
order = invoices.sort_values(["client_id", "invoice_date"])
position = order.groupby("client_id").cumcount()
n_invoices = order.groupby("client_id")["client_id"].transform("size")
is_late = position >= n_invoices / 2
early_mean = order[~is_late].groupby("client_id")["total_cons"].mean()
late_mean = order[is_late].groupby("client_id")["total_cons"].mean()
X["cons_change"] = ((late_mean - early_mean) / (early_mean + 1)).reindex(X.index).fillna(0)  # +1 avoids dividing by zero

In [ ]:
display(fraud_rate_by(pd.cut(X["cons_change"], [-float("inf"), -0.5, -0.1, 0.1, float("inf")],
                             labels=["drop over 50%", "drop 10-50%", "stable (+/-10%)", "increase over 10%"]),
                      "consumption change, later vs earlier invoices"))

In [ ]:
cm = X.select_dtypes(include=["number"]).corr(method="pearson")
sns.heatmap(cm)

## 5. Tuning vs. feature engineering

In [ ]:
# Same client split as for the baseline, now with the engineered features
X_train, X_test = X.loc[train_ids], X.loc[test_ids]

In [ ]:
# FAST_MODE = True  -> quick sanity check: 20% of training clients, 4 random setting combos
# FAST_MODE = False -> full run: all training data, all 36 combos
FAST_MODE = True

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold

# The scaler lives inside the pipeline, so the same scaling is applied to train AND test data
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("lgbm", LGBMClassifier(random_state=42, verbose=-1)),
])
pg = {
    "lgbm__learning_rate": [0.01, 0.1, 0.2, 1],
    "lgbm__n_estimators": [10, 100, 200],
    "lgbm__subsample_for_bin": [1000, 10000, 20000],
}
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

def tune(X_tr):
    """Same settings search for any feature set, so the comparison is fair."""
    X_fit = X_tr.sample(frac=0.2, random_state=42) if FAST_MODE else X_tr
    if FAST_MODE:
        search = RandomizedSearchCV(pipe, pg, n_iter=4, cv=cv, n_jobs=8, scoring="roc_auc", random_state=42)
    else:
        search = GridSearchCV(pipe, pg, cv=cv, n_jobs=8, scoring="roc_auc")
    return search.fit(X_fit, y_train.loc[X_fit.index])

# Attempt 1: tune the settings on the raw baseline columns
search_base = tune(X_base.loc[train_ids])
record("2. Baseline + tuned settings", search_base, X_base)
print("Best settings:", search_base.best_params_, "| cross-validated AUC:", round(search_base.best_score_, 4))

Tuning barely moves the score, so we went into feature engineering instead.

In [ ]:
# Attempt 2: engineered features
model_fe = LGBMClassifier(random_state=42, verbose=-1).fit(X_train, y_train)
record("3. Engineered features, default settings", model_fe, X)

gscv = tune(X_train)
record("4. Engineered features + tuned settings", gscv, X)
print("Best settings:", gscv.best_params_, "| cross-validated AUC:", round(gscv.best_score_, 4))

## 6. Results

In [ ]:
results_table = pd.DataFrame(results).set_index("model").round(4)
display(results_table)
# A big gap between train and test AUC would mean overfitting; both low would mean underfitting
results_table.plot.bar(rot=15, figsize=(9, 4), ylim=(0.5, 1), title="AUC by model: train vs test")
plt.show()

In [ ]:
y_pred = gscv.predict(X_test)
print("Test AUC:", round(roc_auc_score(y_test, gscv.predict_proba(X_test)[:, 1]), 4))

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["predicted honest", "predicted fraud"], yticklabels=["actually honest", "actually fraud"])
tn, fp, fn, tp = cm.ravel()
print(f"Fraud caught: {tp} of {tp + fn} | honest clients wrongly flagged: {fp} of {fp + tn}")

In [ ]:
from sklearn.metrics import precision_score, recall_score

# Recall: of all clients who really are fraud, how many did the model catch?
# Precision: of all clients the model flagged, how many really are fraud?
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
print(f"Recall: {recall:.1%} ({tp} of {tp + fn} fraud clients caught)")
print(f"Precision: {precision:.1%} ({tp} of {tp + fp} flagged clients really are fraud)")